<a href="https://colab.research.google.com/github/TediBalint/AI-Jegyzetek/blob/master/Algopro/vima/Kettos%20Kompetencia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Szükséges Importok**

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# NE MÓDOSÍTSD: ezzel a seeddel kapod ugyanazt az eredményt minden
# újrafuttatáskor. A szerver determinisztikus.
# ═══════════════════════════════════════════════════════════════════
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset

# --- Reprodukálhatóság: minden véletlent egyetlen seedhez kötünk ---
SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# A feladat megoldásához GPU ajánlott
device = "cuda" if torch.cuda.is_available() else "cpu"
print('device:', device)

## **Fájlok letöltése**

In [ ]:
import os
import gdown

if not os.path.isdir("data"):
    gdown.download_folder(id="1FU9egPtbJYDUKpIBEh8BIV0zwVQ5Zqf5", output="data")

## **Modell architektúra**

In [ ]:
class ConvVAE(nn.Module):
    def __init__(self):
        super(ConvVAE, self).__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Flatten()
        )

        self.enc_mu = nn.Linear(64*8*8, 16)
        self.enc_logvar = nn.Linear(64*8*8, 16)

        self.decoder = nn.Sequential(
            nn.Linear(16, 64*8*8),
            nn.Unflatten(1, (64, 8, 8)),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        enc = self.encoder(x)

        mu = self.enc_mu(enc)
        logvar = self.enc_logvar(enc)

        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)

        z = mu + eps * std

        return self.decoder(z), mu, logvar

## **Fájlok betöltése**

In [ ]:
model = ConvVAE().to(device)
model.load_state_dict(torch.load("data/ae.pt", map_location=device))

transform = transforms.Compose([transforms.Resize((32,32)), transforms.ToTensor()])
cifar_labeled_ds = datasets.CIFAR10('./data', train=True, download=True, transform=transform)

cifar_tensors = torch.stack([x for x, _ in cifar_labeled_ds])
cifar_ds = TensorDataset(cifar_tensors)

## **Itt kezdődik a te kódod**

In [ ]:
TEAM = "" # írd ide a neved

# a meglévő modellt tanítsd, ez kerül beküldésre

In [ ]:
torch.save(model.state_dict(), "submission.pt")

## **Beküldés**

In [ ]:
import requests

r = requests.post(
    f"https://comp.vima.run/submit/task2",
    data={"team": TEAM},
    files={"file": open("submission.pt", "rb")},
    timeout=60,
)
r.json()